# 02 Document Processing, Parsing & Ingestion Pipelines

**Real-World Scenario**: Multi-Vendor Procurement Contract & Invoice Deduplication System.

This notebook demonstrates **MinHash LSH Deduplication** to prune duplicate invoice revisions before indexing, saving embedding costs and vector store RAM, completed by an **End-to-End GenAI LLM Synthesis Call**.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Realistic Multi-Vendor Invoices Dataset Setup

In [ ]:
invoices = [
    {"id": "inv_1001", "vendor": "Acme Corp", "text": "Invoice 1001 total amount 5000 USD for cloud consulting services rendered in January 2026."},
    {"id": "inv_1001_v2", "vendor": "Acme Corp", "text": "Invoice 1001 total amount 5000 USD for cloud consulting services rendered in January 2026 revised."},
    {"id": "inv_2004", "vendor": "Beta Global", "text": "Invoice 2004 total amount 12500 USD for hardware server rack shipment in February 2026."}
]


## Step 3: MinHash LSH Jaccard Similarity Deduplication Scanner

In [ ]:
def compute_jaccard_similarity(text1: str, text2: str) -> float:
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    intersection = len(words1.intersection(words2))
    union = len(words1.union(words2))
    return intersection / union if union > 0 else 0.0

unique_invoices = []
for inv in invoices:
    is_duplicate = False
    for existing in unique_invoices:
        sim = compute_jaccard_similarity(inv["text"], existing["text"])
        if sim >= 0.80:
            print(f"PRUNING DUPLICATE: '{inv['id']}' vs '{existing['id']}' (Similarity: {sim*100:.1f}%)")
            is_duplicate = True
            break
    if not is_duplicate:
        unique_invoices.append(inv)
        print(f"ACCEPTED UNIQUE: '{inv['id']}'")

print(f"\nDeduplication Complete! Ingesting {len(unique_invoices)} of {len(invoices)} invoices.")


## Step 4: Indexing Deduplicated Records into Hidden `.vectordb/` Directory

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

if os.getenv("OPENAI_API_KEY"):
    texts = [inv["text"] for inv in unique_invoices]
    metadatas = [{"id": inv["id"], "vendor": inv["vendor"]} for inv in unique_invoices]
    
    vectorstore = FAISS.from_texts(texts, OpenAIEmbeddings(), metadatas=metadatas)
    save_path = os.path.join(".vectordb", "02_ingestion_index")
    vectorstore.save_local(save_path)
    print(f"=== Deduplicated Ingested Vectors Saved to Hidden Folder: {save_path} ===")


## Step 5: Executing Complete GenAI Procurement Query & LLM Synthesis Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    save_path = os.path.join(".vectordb", "02_ingestion_index")
    embeddings = OpenAIEmbeddings()
    loaded_vectorstore = FAISS.load_local(save_path, embeddings, allow_dangerous_deserialization=True)
    retriever = loaded_vectorstore.as_retriever(search_kwargs={"k": 2})
    
    prompt = ChatPromptTemplate.from_template("Summarize the invoice details strictly using context:\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    query = "What is the total amount and service rendered in Invoice 1001?"
    docs = retriever.invoke(query)
    context_text = "\n".join([d.page_content for d in docs])
    
    response = llm.invoke(prompt.format(context=context_text, question=query))
    print("=== Complete Procurement GenAI LLM Response ===")
    print(response.content)
